This is a data I manually extracted from jiji website property page specifically for Addis Abeba location in order to generate the housing price .csv for rent and buying, why manually? simply the site can't be scrapped!

In [4]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

# Number of records (adjust as needed)
N_RECORDS = 15000

# Locations from Jiji data (with relative weights and price multipliers)
LOCATIONS = {
    'Bole': {'weight': 0.610, 'multiplier': 1.3},
    'Yeka': {'weight': 0.211, 'multiplier': 1.0},
    'Akaky Kaliti': {'weight': 0.058, 'multiplier': 0.7},
    'Nifas Silk-Lafto': {'weight': 0.050, 'multiplier': 0.85},
    'Kirkos': {'weight': 0.027, 'multiplier': 1.1},
    'Arada': {'weight': 0.013, 'multiplier': 0.9},
    'Lideta': {'weight': 0.012, 'multiplier': 0.7},
    'Kolfe Keranio': {'weight': 0.009, 'multiplier': 0.75},
    'Gullele': {'weight': 0.006, 'multiplier': 0.7},
    'Addis Ketema': {'weight': 0.004, 'multiplier': 0.7},
}

# Property types from Jiji data (with relative weights)
PROPERTY_TYPES = {
    'House': {'weight': 0.38, 'area_range': (100, 400), 'stories_range': (1, 3), 'bedroom_multiplier': 1.0},
    'Apartment': {'weight': 0.27, 'area_range': (40, 180), 'stories_range': (1, 4), 'bedroom_multiplier': 0.7},
    'Condo': {'weight': 0.17, 'area_range': (50, 150), 'stories_range': (1, 3), 'bedroom_multiplier': 0.65},
    'Villa': {'weight': 0.14, 'area_range': (200, 600), 'stories_range': (1, 3), 'bedroom_multiplier': 1.2},
    'Studio': {'weight': 0.04, 'area_range': (20, 50), 'stories_range': (1, 1), 'bedroom_multiplier': 0.5},
}

# Furnishing distribution (based on Jiji data)
FURNISHING = ['furnished', 'semi-furnished', 'unfurnished']
FURNISHING_WEIGHTS = [0.477, 0.168, 0.355]  # from Jiji sale data

# Condition distribution (based on Jiji data - 'House' condition used as overall proxy)
CONDITIONS = ['Newly-Built', 'Fairly Used', 'Uncompleted Building', 'Old', 'Under Construction', 'Renovated', 'Off-Plan']
CONDITION_WEIGHTS = [0.682, 0.25, 0.022, 0.022, 0.013, 0.006, 0.005]  # normalized from Jiji data

# Categorical features with probabilities
MAINROAD_PROB = 0.75
GUESTROOM_PROB = 0.25
BASEMENT_PROB = 0.15
HOTWATER_PROB = 0.35
AIRCONDITIONING_PROB = 0.55
PREFAREA_PROB = 0.30
PARKING_RANGE = (0, 5)

# Base price per sqm (in ETB)
BASE_PRICE_PER_SQM = 80000

# Price noise (random variation)
PRICE_NOISE_STD = 0.15


Helper Functions

In [5]:
def weighted_random(choices, weights):
    """Return a random choice based on weights."""
    return random.choices(choices, weights=weights, k=1)[0]

def generate_area(property_type):
    """Generate area (sqm) based on property type."""
    area_min, area_max = PROPERTY_TYPES[property_type]['area_range']
    # Make area more realistic: smaller apartments tend to be smaller
    if property_type == 'Apartment':
        # More apartments in the 60-120 range
        return int(np.random.triangular(area_min, (area_min+area_max)/2, area_max))
    elif property_type == 'House':
        return int(np.random.triangular(area_min, (area_min+area_max)/2, area_max))
    else:
        return int(np.random.uniform(area_min, area_max))

def generate_bedrooms(area, property_type):
    """Generate bedrooms based on area and property type."""
    # Base bedrooms per area
    if property_type in ['Apartment', 'Condo']:
        if area < 50:
            return 1
        elif area < 80:
            return 2
        elif area < 120:
            return 3
        elif area < 160:
            return 4
        else:
            return 5
    elif property_type == 'Studio':
        return 1
    else:  # House, Villa
        if area < 100:
            return 2
        elif area < 150:
            return 3
        elif area < 200:
            return 4
        elif area < 300:
            return 5
        else:
            return 6
    # Add random variation
    # return int(np.random.normal(base_bedrooms, 0.5).round())

def generate_bathrooms(bedrooms):
    """Generate bathrooms based on bedrooms."""
    if bedrooms == 1:
        return 1
    elif bedrooms == 2:
        return random.choice([1, 2])
    elif bedrooms == 3:
        return random.choice([2, 3])
    elif bedrooms == 4:
        return random.choice([2, 3, 4])
    else:  # 5+ bedrooms
        return random.choice([3, 4, 5])

def generate_stories(property_type, area):
    """Generate number of stories based on property type."""
    if property_type == 'Apartment':
        return random.choice([1, 2, 3, 4])
    elif property_type == 'Condo':
        return random.choice([1, 2, 3])
    elif property_type == 'Villa':
        return random.choice([1, 2, 3])
    elif property_type == 'House':
        return random.choice([1, 2])
    else:  # Studio
        return 1

def generate_price(area, location, property_type, bedrooms, bathrooms):
    """Generate price in ETB based on property features."""
    # Base price per sqm depends on location
    location_multiplier = LOCATIONS[location]['multiplier']
    
    # Adjust for property type
    property_multiplier = {
        'House': 1.0,
        'Apartment': 0.85,
        'Condo': 0.8,
        'Villa': 1.3,
        'Studio': 0.6,
    }[property_type]
    
    # Adjust for bedrooms and bathrooms
    bedroom_bonus = (bedrooms - 1) * 0.05
    bathroom_bonus = (bathrooms - 1) * 0.03
    
    # Calculate base price
    base_price = area * BASE_PRICE_PER_SQM
    adjusted_price = base_price * location_multiplier * property_multiplier * (1 + bedroom_bonus + bathroom_bonus)
    
    # Add random noise
    noise = np.random.normal(1, PRICE_NOISE_STD)
    final_price = int(adjusted_price * noise)
    
    # Ensure price is within realistic bounds
    min_price = 500000
    max_price = 150000000
    return max(min_price, min(max_price, final_price))

def generate_categorical(prob):
    """Return 'yes' with given probability, else 'no'."""
    return 'yes' if random.random() < prob else 'no'

def generate_furnishing():
    """Generate furnishing status based on Jiji distribution."""
    return weighted_random(FURNISHING, FURNISHING_WEIGHTS)

def generate_condition():
    """Generate condition based on Jiji distribution."""
    return weighted_random(CONDITIONS, CONDITION_WEIGHTS)

def generate_parking():
    """Generate number of parking spaces."""
    return int(np.random.triangular(0, 2, 5))


Generating the Dataset

In [6]:
def generate_dataset(n_records=N_RECORDS):
    """Generate a synthetic Ethiopian housing dataset."""
    
    records = []
    location_keys = list(LOCATIONS.keys())
    location_weights = [LOCATIONS[loc]['weight'] for loc in location_keys]
    
    for _ in range(n_records):
        # Select location based on Jiji distribution
        location = weighted_random(location_keys, location_weights)
        
        # Select property type based on Jiji distribution
        property_type = weighted_random(
            list(PROPERTY_TYPES.keys()),
            [PROPERTY_TYPES[pt]['weight'] for pt in PROPERTY_TYPES]
        )
        
        # Generate features
        area = generate_area(property_type)
        bedrooms = generate_bedrooms(area, property_type)
        bathrooms = generate_bathrooms(bedrooms)
        stories = generate_stories(property_type, area)
        
        # Generate categorical features
        mainroad = generate_categorical(MAINROAD_PROB)
        guestroom = generate_categorical(GUESTROOM_PROB)
        basement = generate_categorical(BASEMENT_PROB)
        hotwaterheating = generate_categorical(HOTWATER_PROB)
        airconditioning = generate_categorical(AIRCONDITIONING_PROB)
        prefarea = generate_categorical(PREFAREA_PROB)
        
        # Generate other features
        furnishingstatus = generate_furnishing()
        parking = generate_parking()
        condition = generate_condition()
        
        # Generate price
        price = generate_price(area, location, property_type, bedrooms, bathrooms)
        
        # Create record (in Kaggle Housing.csv format with location added)
        record = {
            'price': price,
            'area': area,
            'bedrooms': bedrooms,
            'bathrooms': bathrooms,
            'stories': stories,
            'mainroad': mainroad,
            'guestroom': guestroom,
            'basement': basement,
            'hotwaterheating': hotwaterheating,
            'airconditioning': airconditioning,
            'parking': parking,
            'prefarea': prefarea,
            'furnishingstatus': furnishingstatus,
            'condition': condition,
            'location': location,  # Added for Ethiopian context
        }
        
        records.append(record)
    
    return pd.DataFrame(records)


Data Cleaning and Exporing

In [7]:
def clean_dataset(df):
    """Clean the dataset to ensure quality."""
    # Remove any rows with missing values
    df = df.dropna()
    
    # Ensure all values are within reasonable bounds
    df = df[df['price'] >= 100000]
    df = df[df['price'] <= 200000000]
    df = df[df['area'] >= 10]
    df = df[df['area'] <= 1500]
    df = df[df['bedrooms'] >= 0]
    df = df[df['bedrooms'] <= 10]
    df = df[df['bathrooms'] >= 0]
    df = df[df['bathrooms'] <= 8]
    
    # Reset index
    df = df.reset_index(drop=True)
    
    return df

# Generate and clean data
print("🚀 Generating Ethiopian housing dataset...")
df = generate_dataset(N_RECORDS)

print(f"📊 Generated {len(df)} records.")
print("\n📋 First 5 records:")
print(df.head())

print("\n📈 Data Summary:")
print(df.describe())

print("\n📍 Location Distribution:")
print(df['location'].value_counts(normalize=True).round(3))

print("\n🏠 Property Type Distribution:")
print(df['property_type'].value_counts(normalize=True).round(3) if 'property_type' in df.columns else "Not tracked")

# Save to CSV
csv_filename = 'ethiopian_housing_data.csv'
df.to_csv(csv_filename, index=False)
print(f"\n✅ Data saved to '{csv_filename}'")

# Optional: Save as Excel
# df.to_excel('ethiopian_housing_data.xlsx', index=False)

print("\n📁 File ready for your ML pipeline!")

🚀 Generating Ethiopian housing dataset...
📊 Generated 15000 records.

📋 First 5 records:
      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  21067153   229         5          4        1      yes        no       no   
1  22415522   183         4          2        2      yes        no       no   
2  12892675   151         4          2        2      yes        no       no   
3  33632338   208         5          5        2      yes        no      yes   
4  11240516   139         4          3        2      yes       yes       no   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  \
0              no             yes        4       no        furnished   
1             yes              no        1       no        furnished   
2              no              no        3       no      unfurnished   
3             yes             yes        4      yes        furnished   
4              no              no        1       no        furnished   

   

(15000, 15)